# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record set @id's
print("Available record sets:")
for record_set in metadata.record_sets:
    print(f"- @id: {record_set.id}, name: {record_set.name}")

print("\nRecord sets and their fields:")
for record_set in metadata.record_sets:
    print(f"\nRecord set: {record_set.name} (@id: {record_set.id})")
    for field in record_set.fields:
        field_type = getattr(field, 'data_type', None)
        print(f"    - Field @id: {field.id}\n      name: {field.name}\n      type: {field_type}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# List of record set @id's to extract
record_sets = [rs.id for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for {record_set_id}")
    except Exception as exc:
        print(f"Failed to load {record_set_id}: {exc}")

# Show columns for a chosen record set
chosen_record_set = record_sets[0] if record_sets else None
if chosen_record_set:
    print(f"\nColumns for {chosen_record_set}:")
    print(dataframes[chosen_record_set].columns.tolist())
    dataframes[chosen_record_set].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example: Select a numeric field for analysis
# Please refer to the printed overview output above to find relevant field @id's.
# For demonstration, attempt to find a numeric field (e.g. coverage or MW) in the columns

record_set_id = chosen_record_set  # The selected record set
df = dataframes.get(record_set_id)

# Attempt to automatically choose a likely numeric field
numeric_candidate_fields = [col for col in df.columns if any(s in col.lower() for s in ['coverage', 'mw', 'abundance', 'count', 'peptide'])]
numeric_field = numeric_candidate_fields[0] if numeric_candidate_fields else df.select_dtypes(include='number').columns[0]
print(f"Using numeric field: {numeric_field}")

threshold = df[numeric_field].quantile(0.75)
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold} (75th percentile):")
print(filtered_df.head())

filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try to find a grouping field (e.g. 'description', 'accession', or 'sample')
possible_group_fields = [col for col in df.columns if any(w in col.lower() for w in ['description', 'accession', 'sample', 'group'])]
if possible_group_fields:
    group_field = possible_group_fields[0]
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
    print(grouped_df.head())
else:
    print("No appropriate group field found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Histogram of the selected numeric field
plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True, color='skyblue')
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If a group field was found, visualize grouped means
if 'grouped_df' in locals():
    plt.figure(figsize=(10,6))
    sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_field}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded the Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells dataset using `mlcroissant` via the Croissant schema.
- Using the schema, we explored record sets, their fields, and extracted records by referencing their `@id`s.
- We performed basic filtering and normalization on a key numeric field and visualized distributions.
- The Croissant schema makes it easy to reliably identify and manipulate entities in a FAIR-compliant manner.